In [0]:
%python
# spark.conf.set("fs.azure.account.key.<<storageacc-name>>.dfs.core.windows.net", "<<yourkey>>")
# must be used only if you did not link Databricks and Azure storage using access connector
# bad practice to use this approach, leaking key in code. 
# good practice to use access connector , managed identity through unity catalog

In [0]:
SHOW CATALOGS

catalog
samples
system
workspace


In [0]:
-- sql comments , you can't use # in sql 
-- SHIFT + ENTER
SHOW DATABASES

databaseName
bronze
default
information_schema
silver


In [0]:
SHOW DATABASES in workspace 
-- workspace is catalog name

databaseName
bronze
default
information_schema
silver


In [0]:
SHOW   TABLES in default 

database,tableName,isTemporary
,_sqldf,true


In [0]:
CREATE DATABASE IF NOT EXISTS orderdb; -- will be created in default catalog

In [0]:
SHOW DATABASES

databaseName
bronze
default
information_schema
orderdb
silver


In [0]:
-- create a table in gold storage, we call it as orders order_id, customer_id, product_id are INt, amount as float
-- azure adls storage, gold zone, in folder orders
-- DELTA TABLE USES PARQUET as format
CREATE TABLE IF NOT EXISTS orderdb.orders (order_id INT, customer_id INT, product_id INT, amount FLOAT)
    USING DELTA 
--    LOCATION 'abfss://gold@gksdatalake4.dfs.core.windows.net/orders'
 
-- Location is optional, don't use location for free edition 
-- for azure databricks abfss://gold@gksdatalake4.dfs.core.windows.net/orders

In [0]:
DESCRIBE EXTENDED orderdb.orders

col_name,data_type,comment
order_id,int,null
customer_id,int,null
product_id,int,null
amount,float,null
,,
# Detailed Table Information,,
Catalog,workspace,
Database,orderdb,
Table,orders,
Created Time,Tue Apr 21 01:36:44 UTC 2026,


In [0]:
DESCRIBE DETAIL workspace.orderdb.orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,e7c1bbf8-8bd9-4165-9677-129aef12c63c,workspace.orderdb.orders,null,,2026-04-21T01:36:42.521Z,2026-04-21T01:36:44.000Z,List(),List(),0,0,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-d3888d93-cd49-4d89-b5de-d3fc45a3f97f, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8f06ccf6-8798-4387-a990-721c96d8aaf9)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
-- insert 1 record, observe orders directory, _delta_log directory
-- WRITE OPERATION, INSERT, 
-- ADD Records
INSERT INTO orderdb.orders VALUES (1,10,100,123.4)

num_affected_rows,num_inserted_rows
1,1


In [0]:
-- SELECT must pick records from parquet files which are located in _delta-logs/jsons
-- with add
SELECT * FROM orderdb.orders

order_id,customer_id,product_id,amount
1,10,100,123.4


In [0]:
-- how update and delete can be performed while your original parquet files in delta tables are immutable, no changes are made to the original files
-- no overwrite
-- existing files should not be removed
-- existing files should not be modified
-- COPY ON WRITE OPERATION - COW
-- UPDATE RECORDS , my old files are immutable
-- SYSTEM SHALL COPY OLD FILE CONTENT INTO NEW FILES, while coping, updated subset of the data [where, no where] shall be written with modification, if no change in data, then it is writeen as is
-- SYSTEM SHALL MARK [SOFT DELETE] the old files as deleted, then add the new parquet files

UPDATE orderdb.orders SET amount = 150 WHERE order_id = 1

-- go and check your deltalog json file, check if any new parquet files are created

num_affected_rows
1


In [0]:
SELECT * FROM orderdb.orders

order_id,customer_id,product_id,amount
1,10,100,150.0


In [0]:
-- DELETE THAT ONE RECORDS
-- soft delete, keep the original insert parquet, update parquet files as is
-- 00000...3.json, it must mark the update parquet file as deleted
DELETE FROM orderdb.orders WHERE order_id = 1 

num_affected_rows
1


In [0]:
SELECT * FROM orderdb.orders

order_id,customer_id,product_id,amount


In [0]:
-- wanted to see history on orders table
-- create table, insert 1 record, update 1 record, delete 1 record
DESCRIBE HISTORY orderdb.orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2025-09-17T05:23:55Z,1375526870449108,gs@training.sh,DELETE,"Map(predicate -> [""(order_id#32505 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 597, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 480, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 117)",null,Databricks-Runtime/15.4.x-scala2.12
2,2025-09-17T05:09:50Z,1375526870449108,gs@training.sh,UPDATE,"Map(predicate -> [""(order_id#23060 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1041, numDeletionVectorsUpdated -> 0, scanTimeMs -> 498, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1360, rewriteTimeMs -> 543)",null,Databricks-Runtime/15.4.x-scala2.12
1,2025-09-17T04:46:27Z,1375526870449108,gs@training.sh,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1360, numOutputRows -> 1)",null,Databricks-Runtime/15.4.x-scala2.12
0,2025-09-17T04:36:56Z,1375526870449108,gs@training.sh,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,null,WriteSerializable,true,Map(),null,Databricks-Runtime/15.4.x-scala2.12


In [0]:
-- so far, version 3 is latest, current, no records, deleted all
-- I want to inspect what was the data before delete, we don't roll back, means, gksdb.orders remain with latest ie no records

-- INSPECT OLD VERSION WITHOUT CHANGES CURRENT/LATEST VERSION 3

-- 2 means the update version
SELECT * FROM orderdb.orders VERSION AS OF 0

order_id,customer_id,product_id,amount


In [0]:
-- sometimes knowing exact version on tables will be difficult
-- sometimes you know time, when you deploy the new code, rollout new release...
-- copy paste time from your history, update one
-- you need to change time stamp to reflect latest time
SELECT * FROM orderdb.orders TIMESTAMP AS OF  '2025-09-17T05:09:50.000+00:00'

order_id,customer_id,product_id,amount
1,10,100,150.0


In [0]:
%python
# how to read this version as of using dataframe
# insert version, before update, delete
# no don't read like a table from hive meta catalog
# but read as delta files
# fix the path correctly, will not work for free edition
df = spark.read.format("delta").option("versionAsOf", 1).load("abfss://gold@gksdatalake4.dfs.core.windows.net/orders")
df.printSchema()
df.show() # 1 record insert version

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- amount: float (nullable = true)

+--------+-----------+----------+------+
|order_id|customer_id|product_id|amount|
+--------+-----------+----------+------+
|       1|         10|       100| 123.4|
+--------+-----------+----------+------+



In [0]:
-- ROLLBACK
-- RESTORE / version / timestamp
-- whatever we restore that will become latest version
-- rollback to the Update version, just before delete,, amount is 150
RESTORE TABLE orderdb.orders VERSION AS OF 2 

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1360,1,0,1,0,1360


In [0]:
SELECT * FROM orderdb.orders

order_id,customer_id,product_id,amount
1,10,100,150.0


In [0]:
DESCRIBE HISTORY orderdb.orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2025-09-17T06:10:00Z,1375526870449108,gs@training.sh,RESTORE,"Map(timestamp -> null, version -> 2)",null,List(3457151089570224),0915-075854-nf6tokeh,3,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 0, numRemovedFiles -> 0, restoredFilesSize -> 1360, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 1360)",null,Databricks-Runtime/15.4.x-scala2.12
3,2025-09-17T05:23:55Z,1375526870449108,gs@training.sh,DELETE,"Map(predicate -> [""(order_id#32505 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 597, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 480, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 117)",null,Databricks-Runtime/15.4.x-scala2.12
2,2025-09-17T05:09:50Z,1375526870449108,gs@training.sh,UPDATE,"Map(predicate -> [""(order_id#23060 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1041, numDeletionVectorsUpdated -> 0, scanTimeMs -> 498, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1360, rewriteTimeMs -> 543)",null,Databricks-Runtime/15.4.x-scala2.12
1,2025-09-17T04:46:27Z,1375526870449108,gs@training.sh,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1360, numOutputRows -> 1)",null,Databricks-Runtime/15.4.x-scala2.12
0,2025-09-17T04:36:56Z,1375526870449108,gs@training.sh,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,null,WriteSerializable,true,Map(),null,Databricks-Runtime/15.4.x-scala2.12


In [0]:
-- insert some random data to create more files
INSERT INTO orderdb.orders VALUES (7,7,7,7.0), (6,6,6,6.0), (4,4,4,4.0), (3,3,3,3.0)

num_affected_rows,num_inserted_rows
4,4


In [0]:
-- OPTIMZIE 
-- we have tiny small example, wiht 1 records, 1 partitions
-- in datalake, part-0000-blah-blah..
-- in larger data, 200 partitions, 1000 partitions, if we insert/update/delete, it will create 200/1000 parquet files per action/job
-- some parquet file may have 1 record, some may have 1000 records, some may have millions of records
-- 10000 parquet files, 9000 files has 1 or 2 records, 900 files has 5 records, 100 files - 50 records, total records may be less than 15000 - TOO MANY SMALLER FILES - read all these TINY files, and extract records TAKES TIME

-- OPTIMIZE used to put all these smaller records into 1 big parquet file

-- OPTIMIZE TO COMPACT THE FILES, move smaller records into bigger file
-- plain optmizer does not compact for performance, means, it does not group/co-group records for read efficiency
-- more like a random records, stored together, not like a group of records belong to same country

OPTIMIZE orderdb.orders; 

-- 9 files content now copied into 1 single file, improve i/o performance

path,metrics
abfss://gold@gksdatalake4.dfs.core.windows.net/orders,"List(1, 2, List(1424, 1424, 1424.0, 1, 1424), List(1360, 1408, 1384.0, 2, 2768), 0, null, null, 0, 1, 2, 0, true, 0, 0, 1758090441965, 1758090444947, 4, 1, null, List(0, 0), 4, 4, 152, 0, null)"


In [0]:
DESCRIBE HISTORY orderdb.orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2025-09-17T06:27:23Z,1375526870449108,gs@training.sh,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3457151089570224),0915-075854-nf6tokeh,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2768, p25FileSize -> 1424, numDeletionVectorsRemoved -> 0, minFileSize -> 1424, numAddedFiles -> 1, maxFileSize -> 1424, p75FileSize -> 1424, p50FileSize -> 1424, numAddedBytes -> 1424)",null,Databricks-Runtime/15.4.x-scala2.12
5,2025-09-17T06:21:21Z,1375526870449108,gs@training.sh,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1408, numOutputRows -> 4)",null,Databricks-Runtime/15.4.x-scala2.12
4,2025-09-17T06:10:00Z,1375526870449108,gs@training.sh,RESTORE,"Map(timestamp -> null, version -> 2)",null,List(3457151089570224),0915-075854-nf6tokeh,3,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 0, numRemovedFiles -> 0, restoredFilesSize -> 1360, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 1360)",null,Databricks-Runtime/15.4.x-scala2.12
3,2025-09-17T05:23:55Z,1375526870449108,gs@training.sh,DELETE,"Map(predicate -> [""(order_id#32505 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 597, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 480, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 117)",null,Databricks-Runtime/15.4.x-scala2.12
2,2025-09-17T05:09:50Z,1375526870449108,gs@training.sh,UPDATE,"Map(predicate -> [""(order_id#23060 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1041, numDeletionVectorsUpdated -> 0, scanTimeMs -> 498, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1360, rewriteTimeMs -> 543)",null,Databricks-Runtime/15.4.x-scala2.12
1,2025-09-17T04:46:27Z,1375526870449108,gs@training.sh,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1360, numOutputRows -> 1)",null,Databricks-Runtime/15.4.x-scala2.12
0,2025-09-17T04:36:56Z,1375526870449108,gs@training.sh,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,null,WriteSerializable,true,Map(),null,Databricks-Runtime/15.4.x-scala2.12


In [0]:
-- SIMPLE OPTIMIZE will not group the data based on semantics like keep all product id together
-- OPTIMISE WITH ZORDER BY will group the data based on semantic, product_id, country code, category etc
-- if your query has WHERE product_id or group by product_id, join with other tables, where keys are used

OPTIMIZE gks_hive.orders ZORDER BY (customer_id);

-- OPTIMIZE gksdb.orders ZORDER BY (product_id);
-- optimize not done any change here, becuase we have all content in 1 file

path,metrics
abfss://gold@gksdatalake4.dfs.core.windows.net/orders,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1424), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1758090967582, 1758090968876, 4, 0, null, List(0, 0), 4, 4, 0, 0, null)"


In [0]:
DESCRIBE HISTORY gks_hive.orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2025-09-17T06:27:23Z,1375526870449108,gs@training.sh,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3457151089570224),0915-075854-nf6tokeh,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2768, p25FileSize -> 1424, numDeletionVectorsRemoved -> 0, minFileSize -> 1424, numAddedFiles -> 1, maxFileSize -> 1424, p75FileSize -> 1424, p50FileSize -> 1424, numAddedBytes -> 1424)",null,Databricks-Runtime/15.4.x-scala2.12
5,2025-09-17T06:21:21Z,1375526870449108,gs@training.sh,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1408, numOutputRows -> 4)",null,Databricks-Runtime/15.4.x-scala2.12
4,2025-09-17T06:10:00Z,1375526870449108,gs@training.sh,RESTORE,"Map(timestamp -> null, version -> 2)",null,List(3457151089570224),0915-075854-nf6tokeh,3,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 0, numRemovedFiles -> 0, restoredFilesSize -> 1360, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 1360)",null,Databricks-Runtime/15.4.x-scala2.12
3,2025-09-17T05:23:55Z,1375526870449108,gs@training.sh,DELETE,"Map(predicate -> [""(order_id#32505 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 597, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 480, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 117)",null,Databricks-Runtime/15.4.x-scala2.12
2,2025-09-17T05:09:50Z,1375526870449108,gs@training.sh,UPDATE,"Map(predicate -> [""(order_id#23060 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1041, numDeletionVectorsUpdated -> 0, scanTimeMs -> 498, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1360, rewriteTimeMs -> 543)",null,Databricks-Runtime/15.4.x-scala2.12
1,2025-09-17T04:46:27Z,1375526870449108,gs@training.sh,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1360, numOutputRows -> 1)",null,Databricks-Runtime/15.4.x-scala2.12
0,2025-09-17T04:36:56Z,1375526870449108,gs@training.sh,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,null,WriteSerializable,true,Map(),null,Databricks-Runtime/15.4.x-scala2.12


In [0]:
-- 3 months after our work, we have history of old files
-- you know that we have all old old parquet still remains.
-- too many files, too large cost money in data lake
-- remove older files, DAMAGE, if we remove older files, we cannot go back to history, timetravel (SELECT ..as of), RESTORE not possible

-- DATABRICKS has default 7 days policy, that you cannot remove the content older than 7 days
-- if RETAIN 14 DAYS; not mentioned, then it takes  7 days as default
-- remove the files older than 14 days, but we don't any such files
VACUUM gks_hive.orders RETAIN 336 HOURS;
-- no change

com.databricks.backend.common.rpc.SparkDriverExceptions$SQLExecutionException: java.lang.IllegalArgumentException: requirement failed: Are you sure you would like to vacuum files with such a low retention period? If you have
writers that are currently writing to this table, there is a risk that you may corrupt the
state of your Delta table.
If you are certain that there are no operations being performed on this table, such as
insert/upsert/delete/optimize, then you may turn off this check by setting:
spark.databricks.delta.retentionDurationCheck.enabled = false
If you are not sure, please use a value not less than "168 hours".
       
	at scala.Predef$.require(Predef.scala:281)
	at com.databricks.sql.transaction.tahoe.commands.VacuumCommand$.checkRetentionPeriodSafety(VacuumCommand.scala:113)
	at com.databricks.sql.transaction.tahoe.commands.VacuumCommand$.getValidFilesFromSnapshot(VacuumCommand.scala:133)
	at com.databricks.sql.transaction.tahoe.commands.VacuumCommand$.$anonfun$gc$2(V

In [0]:
DESCRIBE HISTORY gks_hive.orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2025-09-17T06:41:17Z,1375526870449108,gs@training.sh,VACUUM END,Map(status -> COMPLETED),null,List(3457151089570224),0915-075854-nf6tokeh,7,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/15.4.x-scala2.12
7,2025-09-17T06:41:15Z,1375526870449108,gs@training.sh,VACUUM START,"Map(defaultRetentionMillis -> 604800000, retentionCheckEnabled -> true, specifiedRetentionMillis -> 1209600000)",null,List(3457151089570224),0915-075854-nf6tokeh,6,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/15.4.x-scala2.12
6,2025-09-17T06:27:23Z,1375526870449108,gs@training.sh,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3457151089570224),0915-075854-nf6tokeh,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2768, p25FileSize -> 1424, numDeletionVectorsRemoved -> 0, minFileSize -> 1424, numAddedFiles -> 1, maxFileSize -> 1424, p75FileSize -> 1424, p50FileSize -> 1424, numAddedBytes -> 1424)",null,Databricks-Runtime/15.4.x-scala2.12
5,2025-09-17T06:21:21Z,1375526870449108,gs@training.sh,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1408, numOutputRows -> 4)",null,Databricks-Runtime/15.4.x-scala2.12
4,2025-09-17T06:10:00Z,1375526870449108,gs@training.sh,RESTORE,"Map(timestamp -> null, version -> 2)",null,List(3457151089570224),0915-075854-nf6tokeh,3,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 0, numRemovedFiles -> 0, restoredFilesSize -> 1360, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 1360)",null,Databricks-Runtime/15.4.x-scala2.12
3,2025-09-17T05:23:55Z,1375526870449108,gs@training.sh,DELETE,"Map(predicate -> [""(order_id#32505 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 597, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 480, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 117)",null,Databricks-Runtime/15.4.x-scala2.12
2,2025-09-17T05:09:50Z,1375526870449108,gs@training.sh,UPDATE,"Map(predicate -> [""(order_id#23060 = 1)""])",null,List(3457151089570224),0915-075854-nf6tokeh,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1360, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1041, numDeletionVectorsUpdated -> 0, scanTimeMs -> 498, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1360, rewriteTimeMs -> 543)",null,Databricks-Runtime/15.4.x-scala2.12
1,2025-09-17T04:46:27Z,1375526870449108,gs@training.sh,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1360, numOutputRows -> 1)",null,Databricks-Runtime/15.4.x-scala2.12
0,2025-09-17T04:36:56Z,1375526870449108,gs@training.sh,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3457151089570224),0915-075854-nf6tokeh,null,WriteSerializable,true,Map(),null,Databricks-Runtime/15.4.x-scala2.12


In [0]:
-- let us try delete the files less than 7 days ie 5 days, 120 hrs, note: retentionCheckEnabled is true by default
-- ERROR
VACUUM gks_hive.orders RETAIN 120 HOURS;
-- not a syntatical error, low rention period, it not 7 days older

com.databricks.backend.common.rpc.SparkDriverExceptions$SQLExecutionException: java.lang.IllegalArgumentException: requirement failed: Are you sure you would like to vacuum files with such a low retention period? If you have
writers that are currently writing to this table, there is a risk that you may corrupt the
state of your Delta table.
If you are certain that there are no operations being performed on this table, such as
insert/upsert/delete/optimize, then you may turn off this check by setting:
spark.databricks.delta.retentionDurationCheck.enabled = false
If you are not sure, please use a value not less than "168 hours".
       
	at scala.Predef$.require(Predef.scala:281)
	at com.databricks.sql.transaction.tahoe.commands.VacuumCommand$.checkRetentionPeriodSafety(VacuumCommand.scala:113)
	at com.databricks.sql.transaction.tahoe.commands.VacuumCommand$.getValidFilesFromSnapshot(VacuumCommand.scala:133)
	at com.databricks.sql.transaction.tahoe.commands.VacuumCommand$.$anonfun$gc$2(V

In [0]:
-- let us set the retentionDurationCheck enabled to false
-- should not do in production, you try to remove files less than 7 days
SET spark.databricks.delta.retentionDurationCheck.enabled=false; 

key,value
spark.databricks.delta.retentionDurationCheck.enabled,false


In [0]:
-- query runs, 24 hours allowed, but we no data older than 24 hours, but no error shown, we disabled retentioncheck
-- VACUUM gksdb.orders RETAIN 24 HOURS DRY RUN;

VACUUM gks_hive.orders RETAIN 1 HOURS DRY RUN;

path
abfss://gold@gksdatalake4.dfs.core.windows.net/orders/part-00000-f3c2787f-0f2b-4d47-b69c-ad853b50fa38-c000.snappy.parquet
abfss://gold@gksdatalake4.dfs.core.windows.net/orders/deletion_vector_db9257e4-7d36-4259-9c8c-c8b385933e50.bin
abfss://gold@gksdatalake4.dfs.core.windows.net/orders/deletion_vector_473ec268-2170-473d-8075-069339ee05a2.bin


In [0]:
VACUUM gks_hive.orders RETAIN 1 HOURS;  -- remove old files

path
abfss://gold@gksdatalake4.dfs.core.windows.net/orders
